# Semana 1: carga, transformación y reporte de RealFakeNews

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `fauxNeuz/RealFakeNews` desde Hugging Face con `HFDatasetLoader`.
3. Obtener un `DataFrame` de pandas y revisar su estructura.
4. Aplicar cinco transformaciones al `DataFrame`.
5. Generar con `Reportes` un resumen del `DataFrame` **transformado** y guardarlo en un archivo `.txt`.

```
HFDatasetLoader → DataFrame → transformaciones → Reportes → reportes/fake-news.txt
```

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará.

Con `streaming=True` y `max_rows=10_000` se toman **las primeras 10.000 filas** del split, sin descargar el dataset completo (108.032 registros según su ficha). Es una muestra: no es aleatoria y no representa necesariamente la proporción de noticias reales y falsas del dataset completo.

El parámetro `filename` hace que los datos originales, sin transformar, también se guarden como CSV en `src/sincronicas/datos/`.

In [ ]:
loader = HFDatasetLoader(
    dataset_name="fauxNeuz/RealFakeNews",
    split="train",
    max_rows=10_000,
    streaming=True,
    filename="datos/fake-news.csv",
)

## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [ ]:
df = loader.load()
df.head()

## 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [ ]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

## 5. Transformar el DataFrame

Cada transformación parte del resultado de la anterior y se guarda en una variable nueva, así se puede comparar el antes y el después. El resultado final queda en `df_transformado`, que es el que se pasa a la clase de reportes.

### 5.1 Eliminar filas duplicadas

Si una misma noticia aparece varias veces, pesa de más en cualquier conteo o promedio. `drop_duplicates()` deja una sola copia de cada fila repetida.

In [ ]:
df_sin_duplicados = df.drop_duplicates().reset_index(drop=True)

print(f"Filas antes: {len(df)}")
print(f"Filas después: {len(df_sin_duplicados)}")
print(f"Duplicados eliminados: {len(df) - len(df_sin_duplicados)}")

### 5.2 Renombrar las columnas al español

`text` pasa a llamarse `texto` y `label` pasa a llamarse `etiqueta`, para mantener el mismo idioma en todo el proyecto.

In [ ]:
df_renombrado = df_sin_duplicados.rename(columns={"text": "texto", "label": "etiqueta"})

df_renombrado.columns.tolist()

### 5.3 Crear la columna `cantidad_palabras`

Cuenta las palabras de cada texto separándolo por espacios. Es la primera variable numérica del `DataFrame`: permite comparar la extensión de las noticias y hace que el reporte incluya estadísticas descriptivas.

In [ ]:
df_con_palabras = df_renombrado.assign(
    cantidad_palabras=df_renombrado["texto"].str.split().str.len()
)

df_con_palabras["cantidad_palabras"].describe()

### 5.4 Filtrar los textos muy cortos

Se conservan solo los textos con **5 palabras o más**. Un texto de una o dos palabras no alcanza para describir una noticia.

In [ ]:
MIN_PALABRAS = 5

df_filtrado = df_con_palabras[df_con_palabras["cantidad_palabras"] >= MIN_PALABRAS].reset_index(drop=True)

print(f"Filas antes del filtro: {len(df_con_palabras)}")
print(f"Filas con {MIN_PALABRAS} palabras o más: {len(df_filtrado)}")

### 5.5 Codificar la etiqueta como número

Se crea `es_falsa`: vale `1` si la noticia es `FAKE` y `0` si es `REAL`. Al ser numérica, su promedio indica directamente la proporción de noticias falsas en la muestra.

In [ ]:
df_transformado = df_filtrado.assign(
    es_falsa=(df_filtrado["etiqueta"] == "FAKE").astype(int)
)

print(df_transformado["etiqueta"].value_counts())
print(f"Proporción de noticias falsas: {df_transformado['es_falsa'].mean():.2%}")
df_transformado.head()

## 6. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [ ]:
from clases_po import Reportes

## 7. Generar el reporte del DataFrame transformado

`generar_reporte(df_transformado)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. Como ahora hay columnas numéricas (`cantidad_palabras` y `es_falsa`), el reporte incluye sus estadísticas descriptivas.

In [ ]:
reportes = Reportes()
texto = reportes.generar_reporte(df_transformado)

print(texto)

## 8. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo queda en `src/sincronicas/reportes/fake-news.txt` y corresponde al `DataFrame` **transformado**.

In [ ]:
ruta = reportes.guardar_reporte(
    df_transformado,
    "reportes/fake-news.txt",
)

print(f"Reporte guardado en: {ruta}")

## 9. Resumen de las transformaciones

| # | Transformación | Operación | Para qué |
|---|----------------|-----------|----------|
| 1 | Eliminar duplicados | `drop_duplicates()` | Que cada noticia cuente una sola vez |
| 2 | Renombrar columnas | `rename()` | Mantener el proyecto en español |
| 3 | Contar palabras | `assign()` + `str.split().str.len()` | Tener una variable numérica de extensión |
| 4 | Filtrar textos cortos | filtro booleano `>= 5` | Descartar textos que no describen una noticia |
| 5 | Codificar la etiqueta | `assign()` + comparación `== "FAKE"` | Calcular la proporción de noticias falsas |

El archivo `reportes/fake-news.txt` describe `df_transformado`, no los datos originales. Los datos originales quedan en `datos/fake-news.csv`.